# Lab 9: Elastic Dislocation — The Forward Problem

> **Colab note:** This notebook is designed to run on **Google Colab**. [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amtseismo/EPS166/blob/main/notebooks/10_forward_modeling_lab.ipynb)

## Introduction

In lecture we established that the relationship between fault slip and surface displacement is **linear** in an elastic half-space. This means that if we know the Green's function — the surface response to unit slip on each fault patch — we can predict the displacement from any slip distribution by simple matrix multiplication:

$$\mathbf{d} = \mathbf{G}\mathbf{s}$$

In this lab you will build that machinery from the ground up using `cutde`, which implements triangular dislocation elements (Meade 2007) equivalent to the Okada (1985) rectangular fault formulation. You will:

1. Construct fault geometries from first principles — computing corner points from strike, dip, and depth
2. Visualize the 3D fault geometry and understand what the triangles represent
3. Compute single-patch Green's functions and build intuition for how the source location controls the surface response
4. Explore how depth, dip, and rake control the character of the surface deformation
5. Prescribe non-uniform slip distributions and observe how they appear at the surface
6. Build the full Green's function matrix for Ridgecrest and compare forward model predictions to real GNSS and InSAR data

## Learning objectives

By the end, you will be able to:

- construct a triangulated fault mesh from geometric parameters
- compute and interpret the Green's function for individual fault patches
- predict surface displacement for any prescribed slip distribution
- explain how depth, dip, rake, and slip heterogeneity affect the surface deformation pattern
- identify the limits of what the surface displacement pattern can reveal about the subsurface slip distribution
- assemble a Green's function matrix for joint GNSS and InSAR prediction

## Notebook outline
- [Setup](#setup)
- [Part I: Building a fault from triangles](#part-i-building-a-fault-from-triangles)
- [Part II: Single-patch Green's functions](#part-ii-single-patch-greens-functions)
- [Part III: Uniform slip and the Okada model](#part-iii-uniform-slip-and-the-okada-model)
- [Part IV: Non-uniform slip distributions](#part-iv-non-uniform-slip-distributions)
- [Part V: Forward prediction for Ridgecrest](#part-v-forward-prediction-for-ridgecrest)
- [Synthesis](#synthesis)
- [Summary](#summary)


## Setup


In [ ]:
%pip install -q cutde

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import cutde.halfspace as hs
import io, urllib.request, pandas as pd
from scipy.optimize import curve_fit

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.titlesize': 11})

# Constants
NU = 0.25            # Poisson's ratio (Poisson solid)
MU = 30e9            # Shear modulus (Pa)
WAVELENGTH = 0.056   # Sentinel-1 C-band wavelength (m)

print('Setup complete.')
print(f'Poisson solid: nu={NU}, mu={MU/1e9:.0f} GPa')


## Part I: Building a fault from triangles

### The geometry of a rectangular fault patch

`cutde` represents fault surfaces as collections of triangles. A rectangular fault patch requires exactly **two triangles** that share a diagonal. Before we can compute anything, we need to derive the coordinates of the four corners of the rectangle from the fault geometry parameters.

Given strike $\phi$, dip $\delta$, and the depth to the top edge $d$, the four corners are:

$$P_0 = C - \frac{L}{2}\hat{\mathbf{a}}, \quad
P_1 = C + \frac{L}{2}\hat{\mathbf{a}}, \quad
P_2 = P_1 + W\hat{\mathbf{d}}, \quad
P_3 = P_0 + W\hat{\mathbf{d}}$$

where $C = (0, 0, -d)$ is the top-center of the fault (we center it at the origin), $\hat{\mathbf{a}}$ is the along-strike unit vector, and $\hat{\mathbf{d}}$ is the down-dip unit vector:

$$\hat{\mathbf{a}} = (\sin\phi,\ \cos\phi,\ 0)$$
$$\hat{\mathbf{d}} = (\cos\delta\cos(\phi+90°),\ -\cos\delta\sin(\phi+90°),\ -\sin\delta)$$

**Coordinate system:** $x$ = East (m), $y$ = North (m), $z$ = Up (m, negative downward).


In [ ]:
def fault_unit_vectors(strike_deg, dip_deg):
    """
    Compute the along-strike and down-dip unit vectors for a fault.

    Parameters
    ----------
    strike_deg : float  Strike in degrees clockwise from North
    dip_deg    : float  Dip in degrees below horizontal

    Returns
    -------
    along_strike : (3,) array   Unit vector along strike (E, N, U)
    down_dip     : (3,) array   Unit vector down dip (E, N, U)
    """
    s = np.radians(strike_deg)
    d = np.radians(dip_deg)
    along_strike = np.array([np.sin(s), np.cos(s), 0.])
    down_dip     = np.array([np.cos(d)*np.cos(s + np.pi/2),
                            -np.cos(d)*np.sin(s + np.pi/2),
                            -np.sin(d)])
    return along_strike, down_dip


def fault_corners(strike_deg, dip_deg, length_km, width_km, depth_top_km):
    """
    Compute the four corners of a rectangular fault patch in meters.
    The fault is centered at the origin horizontally.

    Parameters
    ----------
    strike_deg    : float
    dip_deg       : float
    length_km     : float  Along-strike length
    width_km      : float  Down-dip width
    depth_top_km  : float  Depth to top edge (positive = down)

    Returns
    -------
    P0, P1, P2, P3 : (3,) arrays  Corner coordinates in meters
        P0 = top-left, P1 = top-right, P2 = bottom-right, P3 = bottom-left
    """
    along, downdip = fault_unit_vectors(strike_deg, dip_deg)
    L = length_km * 1e3
    W = width_km  * 1e3
    C = np.array([0., 0., -depth_top_km * 1e3])   # top-center

    P0 = C - L/2 * along
    P1 = C + L/2 * along
    P2 = P1 + W * downdip
    P3 = P0 + W * downdip
    return P0, P1, P2, P3


def fault_triangles(strike_deg, dip_deg, rake_deg,
                    length_km, width_km, depth_top_km, slip_m=1.0):
    """
    Build two triangles and their slip vectors for a rectangular fault patch.

    Returns
    -------
    tris  : (2, 3, 3) array  Triangle vertex coordinates in meters
    slips : (2, 3) array     [strike-slip, dip-slip, tensile] per triangle
    """
    P0, P1, P2, P3 = fault_corners(strike_deg, dip_deg,
                                    length_km, width_km, depth_top_km)
    tris  = np.array([[P0, P1, P2], [P0, P2, P3]], dtype=float)
    r     = np.radians(rake_deg)
    ss    =  slip_m * np.cos(r)   # strike-slip component
    sd    = -slip_m * np.sin(r)   # dip-slip component (cutde convention)
    slips = np.tile([ss, sd, 0.], (2, 1))
    return tris, slips


# ── Task: compute corners for a test fault and print them ──────────────────
# Strike=0° (N-S), dip=45°, depth_top=5km, length=20km, width=10km
P0, P1, P2, P3 = fault_corners(0, 45, 20, 10, 5)

print('Fault corners (x=East, y=North, z=Up, all in meters):')
for name, P in zip(['P0 (top-left)', 'P1 (top-right)',
                    'P2 (bottom-right)', 'P3 (bottom-left)'], [P0,P1,P2,P3]):
    print(f'  {name}: E={P[0]/1e3:+.1f}km  N={P[1]/1e3:+.1f}km  z={P[2]/1e3:+.1f}km')

# Sanity checks
print(f'\nAlong-strike length: {np.linalg.norm(P1-P0)/1e3:.1f} km (expected 20.0)')
print(f'Down-dip width:      {np.linalg.norm(P2-P1)/1e3:.1f} km (expected 10.0)')
print(f'Depth of bottom edge: {-P2[2]/1e3:.1f} km (expected {5+10*np.sin(np.radians(45)):.1f})')


> **Geometry questions:**
> 1. For a vertical fault (dip=90°), what are the down-dip unit vector components? Does this make physical sense?
> 2. For a horizontal fault (dip=0°), what are the down-dip unit vector components?
> 3. Change the strike to 45° and recompute the corners. In which direction does P1 lie relative to P0?
> 4. The depth to the **bottom** edge is not an input parameter but is derived from the geometry. Write an expression for the bottom depth in terms of `depth_top_km`, `width_km`, and `dip_deg`. Check it against the printed value above.


In [ ]:
def visualize_fault_3d(strike_deg, dip_deg, length_km, width_km,
                        depth_top_km, title='', color='steelblue', alpha=0.5,
                        grid_km=40, ax=None):
    """
    Plot a rectangular fault patch in 3D perspective.

    Parameters
    ----------
    strike_deg, dip_deg, length_km, width_km, depth_top_km : fault geometry
    color  : face color for the fault patch
    grid_km: half-width of the surface grid shown for context
    ax     : optional existing 3D axes
    """
    P0, P1, P2, P3 = fault_corners(strike_deg, dip_deg,
                                    length_km, width_km, depth_top_km)
    corners = np.array([P0, P1, P2, P3]) / 1e3  # convert to km

    if ax is None:
        fig = plt.figure(figsize=(9, 7))
        ax  = fig.add_subplot(111, projection='3d')

    # Fault surface
    verts = [[corners[0], corners[1], corners[2], corners[3]]]
    poly  = Poly3DCollection(verts, alpha=alpha, facecolor=color,
                             edgecolor='k', linewidth=1)
    ax.add_collection3d(poly)

    # Top edge (surface trace)
    top = np.array([P0, P1]) / 1e3
    ax.plot(top[:,0], top[:,1], top[:,2], 'r-', lw=3, label='Fault trace')

    # Corner labels
    for name, P in zip(['P0','P1','P2','P3'], corners):
        ax.text(P[0], P[1], P[2], f' {name}', fontsize=8)

    # Surface grid for context
    g = grid_km
    for xi in np.linspace(-g, g, 5):
        ax.plot([xi,xi], [-g,g], [0,0], color='0.85', lw=0.5, zorder=0)
    for yi in np.linspace(-g, g, 5):
        ax.plot([-g,g], [yi,yi], [0,0], color='0.85', lw=0.5, zorder=0)

    depth_bot = -P2[2]/1e3
    ax.set_xlabel('East (km)'); ax.set_ylabel('North (km)')
    ax.set_zlabel('Depth (km)')
    ax.set_xlim(-g, g); ax.set_ylim(-g, g)
    ax.set_zlim(-depth_bot*2, 2)
    ax.invert_zaxis()
    ax.set_title(title or
        f'Strike={strike_deg}°  Dip={dip_deg}°  '
        f'L={length_km}km  W={width_km}km  d={depth_top_km}km')
    ax.legend(fontsize=8)
    return ax


# Visualize three fault geometries side by side
fig = plt.figure(figsize=(16, 5))

configs = [
    dict(strike_deg=0,  dip_deg=90, length_km=30, width_km=15,
         depth_top_km=0, color='steelblue',  title='Vertical strike-slip\n(dip=90°)'),
    dict(strike_deg=0,  dip_deg=45, length_km=30, width_km=20,
         depth_top_km=3, color='firebrick',   title='Normal fault\n(dip=45°)'),
    dict(strike_deg=90, dip_deg=15, length_km=80, width_km=50,
         depth_top_km=5, color='goldenrod',   title='Subduction thrust\n(dip=15°)'),
]

for i, cfg in enumerate(configs):
    ax = fig.add_subplot(1, 3, i+1, projection='3d')
    visualize_fault_3d(**cfg, ax=ax)

plt.suptitle('Three fault geometries — the same triangle framework handles all types',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()


> **Visualization questions:**
> 1. For the subduction thrust (dip=15°), the bottom of the fault is at what depth? Show your calculation.
> 2. What happens to the fault geometry when depth_top_km=0 and dip=90° — does the top edge reach the surface?
> 3. Modify the code to add a fourth fault: a NW-striking (strike=315°) right-lateral fault with dip=85°, length=50km, width=16km, depth_top=0km. This is approximately the Ridgecrest M7.1 geometry. Does it look right?


## Part II: Single-patch Green's functions

### What is a Green's function?

The Green's function for our problem is the surface displacement due to **unit slip** on a **single small fault patch**. By superposition, the displacement from any slip distribution is a weighted sum of Green's functions — one per patch.

In this part, we will:
1. Divide a fault into a grid of sub-patches
2. Compute the Green's function for each patch individually
3. Visualize how the surface response depends on where the patch sits on the fault

This builds the intuition needed to understand why inversion is challenging — different patches produce overlapping surface signals that are hard to disentangle.


In [ ]:
def make_fault_mesh(strike_deg, dip_deg, rake_deg,
                     length_km, width_km, depth_top_km,
                     n_along, n_down, slip_m=1.0):
    """
    Subdivide a rectangular fault into n_along × n_down patches.
    Each patch is represented by two triangles.

    Parameters
    ----------
    n_along : int  Number of patches along strike
    n_down  : int  Number of patches down dip
    slip_m  : float or (n_along, n_down) array  Slip on each patch

    Returns
    -------
    tris  : (2*n_along*n_down, 3, 3)  Triangle vertices in meters
    slips : (2*n_along*n_down, 3)     Slip vectors
    patch_centers : (n_along*n_down, 3)  Center of each patch in meters
    """
    along, downdip = fault_unit_vectors(strike_deg, dip_deg)
    r   = np.radians(rake_deg)
    C0  = np.array([0., 0., -depth_top_km*1e3])
    dl  = length_km*1e3 / n_along
    dw  = width_km*1e3  / n_down

    slip_arr = np.ones((n_along, n_down)) * slip_m if np.isscalar(slip_m) else np.asarray(slip_m)

    all_tris, all_slips, centers = [], [], []
    for i in range(n_along):
        for j in range(n_down):
            corner = C0 + i*dl*along + j*dw*downdip
            P = [corner,
                 corner + dl*along,
                 corner + dl*along + dw*downdip,
                 corner +            dw*downdip]
            s_ij = slip_arr[i, j]
            ss, sd = s_ij*np.cos(r), -s_ij*np.sin(r)
            all_tris += [[P[0],P[1],P[2]], [P[0],P[2],P[3]]]
            all_slips += [[ss,sd,0.], [ss,sd,0.]]
            centers.append(corner + 0.5*dl*along + 0.5*dw*downdip)

    return (np.array(all_tris, dtype=float),
            np.array(all_slips, dtype=float),
            np.array(centers))


def make_obs_grid(grid_km=80, nx=200):
    """Regular surface observation grid, returns (N,3) array and (nx,nx) meshgrids."""
    x = np.linspace(-grid_km*1e3, grid_km*1e3, nx)
    X, Y = np.meshgrid(x, x)
    obs  = np.column_stack([X.ravel(), Y.ravel(), np.zeros(X.size)])
    return obs, X/1e3, Y/1e3


print('Mesh functions defined.')
# Quick check: 3×2 patch mesh
tris, slips, centers = make_fault_mesh(0, 90, 180, 30, 15, 3, 3, 2)
print(f'3×2 mesh: {len(tris)} triangles, {len(centers)} patches')
print(f'First patch center: E={centers[0,0]/1e3:.1f}km  N={centers[0,1]/1e3:.1f}km  z={centers[0,2]/1e3:.1f}km')


In [ ]:
# Compute Green's function for each patch individually
# Use a 4×3 mesh on a right-lateral vertical fault
N_ALONG, N_DOWN = 4, 3
FAULT_PARAMS = dict(strike_deg=0, dip_deg=90, rake_deg=180,
                     length_km=40, width_km=15, depth_top_km=2)

obs, X_km, Y_km = make_obs_grid(grid_km=70, nx=150)

# Build single-patch tris for each patch
_, _, centers = make_fault_mesh(**FAULT_PARAMS, n_along=N_ALONG, n_down=N_DOWN)

along, downdip = fault_unit_vectors(FAULT_PARAMS['strike_deg'], FAULT_PARAMS['dip_deg'])
dl = FAULT_PARAMS['length_km']*1e3 / N_ALONG
dw = FAULT_PARAMS['width_km']*1e3  / N_DOWN
C0 = np.array([0., 0., -FAULT_PARAMS['depth_top_km']*1e3])
r  = np.radians(FAULT_PARAMS['rake_deg'])

# Compute Green's function (E component) for each patch
GF_east = np.zeros((len(obs), N_ALONG*N_DOWN))

for k, (i, j) in enumerate([(i,j) for i in range(N_ALONG) for j in range(N_DOWN)]):
    corner = C0 + i*dl*along + j*dw*downdip
    P = [corner, corner+dl*along,
         corner+dl*along+dw*downdip, corner+dw*downdip]
    tris_k  = np.array([[P[0],P[1],P[2]],[P[0],P[2],P[3]]], dtype=float)
    slips_k = np.tile([np.cos(r), -np.sin(r), 0.], (2,1))  # unit slip
    disp_k  = hs.disp_free(obs, tris_k, slips_k, nu=NU)
    GF_east[:, k] = disp_k[:, 0]  # east component

# Plot Green's functions for all patches
fig, axes = plt.subplots(N_DOWN, N_ALONG, figsize=(14, 8), sharex=True, sharey=True)

vmax = np.percentile(np.abs(GF_east*100), 99)
for k, (i, j) in enumerate([(i,j) for i in range(N_ALONG) for j in range(N_DOWN)]):
    ax = axes[j, i]
    gf = GF_east[:, k].reshape(X_km.shape) * 100  # cm
    ax.contourf(X_km, Y_km, gf, levels=40, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    depth_center = -centers[k,2]/1e3
    ax.set_title(f'Patch ({i+1},{j+1})\nz={depth_center:.1f}km', fontsize=8)
    ax.set_aspect('equal')
    # Mark patch position at surface
    ax.axvline(0, color='k', lw=1, ls='--', alpha=0.5)

for ax in axes[-1,:]:
    ax.set_xlabel('East (km)')
for ax in axes[:,0]:
    ax.set_ylabel('North (km)')

fig.suptitle('Green\'s functions (East surface displacement, cm/m slip)\n'
             'Each panel = response to 1 m slip on one fault patch',
             fontsize=11)
plt.tight_layout()
plt.show()


> **Green's function questions:**
> 1. Compare the shallow patches (row 1) to the deep patches (row 3). How does increasing depth change: (a) the peak amplitude, (b) the spatial extent, and (c) the sharpness of the signal?
> 2. Patches in the same row (same depth) but different along-strike positions produce Green's functions that are shifted along-strike but otherwise similar. Is this what you would expect from the linearity of the elastic equations?
> 3. The Green's functions for adjacent patches overlap substantially at the surface. What does this tell you about the difficulty of resolving slip on individual patches from surface observations?
> 4. If you had a single GNSS station directly above patch (1,1), which other patches could produce a similar signal at that station? How many stations would you need to uniquely identify which patch slipped?


## Part III: Uniform slip and the Okada model

### Recovering familiar results

In the seismic cycle lecture we derived two 1D analytical models — both limiting cases of the Okada elastic dislocation framework:

**Interseismic (Savage-Burford):**
$$v(x) = \frac{V_s}{\pi}\arctan\!\left(\frac{x}{D}\right)$$

**Coseismic (surface rupture):**
$$u(x) = \frac{S}{\pi}\arctan\!\left(\frac{D}{x}\right)$$

Here you will verify that your cutde forward model reproduces these profiles exactly in the appropriate limits.


In [ ]:
# ── Profile extraction helper ─────────────────────────────────────────────────
def fault_profile(strike_deg, dip_deg, rake_deg,
                   length_km, width_km, depth_top_km, slip_m,
                   profile_km=200, n_pts=400,
                   n_along=1, n_down=1):
    """
    Extract a fault-perpendicular surface displacement profile.
    Returns x_km (distance from fault) and displacement (m).
    """
    # Profile runs perpendicular to strike
    perp_deg = strike_deg + 90
    perp_rad = np.radians(perp_deg)
    x_m  = np.linspace(-profile_km*1e3, profile_km*1e3, n_pts)
    obs  = np.column_stack([
        x_m * np.sin(perp_rad),
        x_m * np.cos(perp_rad),
        np.zeros(n_pts)
    ])
    slip_arr = np.ones((n_along, n_down)) * slip_m
    tris, slips, _ = make_fault_mesh(
        strike_deg, dip_deg, rake_deg,
        length_km, width_km, depth_top_km,
        n_along, n_down, slip_m
    )
    disp = hs.disp_free(obs, tris, slips, nu=NU)
    return x_m/1e3, disp


fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: coseismic profile — compare to arctan(D/x) ─────────────────────────
ax = axes[0]
D_list   = [5, 10, 20]
slip_val = 2.0   # m
colors   = plt.cm.viridis(np.linspace(0.15, 0.85, len(D_list)))

for D, color in zip(D_list, colors):
    x_km, disp = fault_profile(0, 90, 180, 500, D, 0, slip_val,
                                 n_along=1, n_down=1)
    # Fault-parallel = north component for N-S strike
    ax.plot(x_km, disp[:,1]*100, color=color, lw=2, label=f'cutde D={D}km')
    # Analytical: u(x) = (S/pi)*arctan(D/x)
    x_nz = np.where(np.abs(x_km) < 0.2, 0.2*np.sign(x_km+1e-9), x_km)
    u_analytic = (slip_val/np.pi) * np.arctan(D / x_nz)
    ax.plot(x_km, u_analytic*100, color=color, lw=1, ls='--')

ax.axvline(0, color='0.4', lw=1, ls='--')
ax.axhline(0, color='0.8', lw=0.5)
ax.set_xlabel('Fault-perpendicular distance (km)')
ax.set_ylabel('Fault-parallel displacement (cm)')
ax.set_title(r'Coseismic: $u(x)=\frac{S}{\pi}\arctan(D/x)$'
             '\nSolid=cutde, Dashed=analytic')
ax.legend(fontsize=8)

# ── Right: interseismic profile — compare to arctan(x/D) ─────────────────────
# Simulate interseismic: deep creep from depth D to infinity
# Approximation: fault locked 0→D, creeping D→very_deep (large W)
ax = axes[1]
Vs = 35.0  # mm/yr

for D, color in zip(D_list, colors):
    # Creeping segment: from D to D+1000km (approximates infinity)
    x_km, disp = fault_profile(0, 90, 0, 500, 1000, D, Vs/1e3,
                                 n_along=1, n_down=1)
    ax.plot(x_km, disp[:,1]*1e3, color=color, lw=2, label=f'cutde D={D}km')
    # Analytical Savage-Burford
    v_analytic = (Vs/np.pi) * np.arctan(x_km / D)
    ax.plot(x_km, v_analytic, color=color, lw=1, ls='--')

ax.axvline(0, color='0.4', lw=1, ls='--')
ax.axhline(0, color='0.8', lw=0.5)
ax.set_xlabel('Fault-perpendicular distance (km)')
ax.set_ylabel('Fault-parallel velocity (mm/yr)')
ax.set_title(r'Interseismic: $v(x)=\frac{V_s}{\pi}\arctan(x/D)$'
             '\nSolid=cutde (deep creep), Dashed=analytic')
ax.legend(fontsize=8)
ax.set_xlim(-150, 150)

plt.suptitle('Okada/cutde recovers familiar 1D analytical results exactly',
             fontsize=12)
plt.tight_layout()
plt.show()


> **Model recovery questions:**
> 1. The cutde result (solid) should match the analytical formula (dashed) closely. Are there any discrepancies? If so, where and why?
> 2. For the coseismic profile, the argument of arctan is $D/x$ — so increasing depth $D$ makes the profile **broader**. Verify this from the plot and explain the physics.
> 3. For the interseismic profile, the argument is $x/D$ — so increasing depth makes the profile **broader**. Is the effect the same or different from the coseismic case?
> 4. In your own words, explain why the coseismic and interseismic profiles have the arctan argument flipped. What physical difference between the two cases produces this mathematical difference?


In [ ]:
# ── Full 2D displacement field for three fault types ──────────────────────────
obs, X_km, Y_km = make_obs_grid(grid_km=70, nx=200)

FAULT_TYPES = [
    dict(strike_deg=0, dip_deg=90, rake_deg=180, slip_m=2.0,
         label='Right-lateral\nstrike-slip'),
    dict(strike_deg=0, dip_deg=60, rake_deg=-90, slip_m=2.0,
         label='Normal fault\n(60° dip)'),
    dict(strike_deg=0, dip_deg=30, rake_deg=90,  slip_m=2.0,
         label='Thrust fault\n(30° dip)'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for j, ft in enumerate(FAULT_TYPES):
    tris, slips, _ = make_fault_mesh(
        ft['strike_deg'], ft['dip_deg'], ft['rake_deg'],
        30, 15, 3, 4, 3, ft['slip_m']
    )
    disp = hs.disp_free(obs, tris, slips, nu=NU)
    U    = disp[:,0].reshape(X_km.shape)*100  # East, cm
    V    = disp[:,1].reshape(X_km.shape)*100  # North, cm
    W    = disp[:,2].reshape(X_km.shape)*100  # Up, cm

    # Horizontal magnitude
    horiz = np.sqrt(U**2 + V**2)
    lim_h = np.percentile(horiz, 98)
    lim_v = max(np.percentile(np.abs(W), 98), 0.1)

    # Top row: vertical displacement + horizontal quivers
    ax = axes[0, j]
    c  = ax.contourf(X_km, Y_km, W, levels=40, cmap='RdBu_r',
                      vmin=-lim_v, vmax=lim_v)
    plt.colorbar(c, ax=ax, label='Up (cm)')
    skip = 15
    ax.quiver(X_km[::skip,::skip], Y_km[::skip,::skip],
              U[::skip,::skip],    V[::skip,::skip],
              scale=lim_h*20, color='k', width=0.003)
    ax.set_title(ft['label'], fontsize=10)
    ax.set_aspect('equal')
    ax.set_xlabel('East (km)')
    if j==0: ax.set_ylabel('North (km) — vertical + horizontal')
    # Mark fault trace
    ax.axvline(0, color='k', lw=2)

    # Bottom row: east component
    ax = axes[1, j]
    lim_e = np.percentile(np.abs(U), 98)
    c2 = ax.contourf(X_km, Y_km, U, levels=40, cmap='RdBu_r',
                      vmin=-lim_e, vmax=lim_e)
    plt.colorbar(c2, ax=ax, label='East (cm)')
    ax.set_aspect('equal')
    ax.set_xlabel('East (km)')
    if j==0: ax.set_ylabel('North (km) — east component')
    ax.axvline(0, color='k', lw=2)

plt.suptitle('Surface displacement for three fault types\n'
             'Strike=0°, depth_top=3km, length=30km, width=15km, slip=2m',
             fontsize=11)
plt.tight_layout()
plt.show()


> **Fault type questions:**
> 1. The normal fault shows a clear asymmetry in vertical displacement between east and west of the fault. Which side is the hanging wall? Does it move up or down?
> 2. The thrust fault produces the opposite vertical pattern from the normal fault. Is the horizontal displacement also reversed, or does it have the same sense?
> 3. The strike-slip fault produces very little vertical displacement. Why? Under what conditions would a strike-slip fault produce significant vertical motion?
> 4. You have only an ascending Sentinel-1 interferogram (LOS ≈ 38° incidence, heading ≈ -12°). For which of these three fault types would the InSAR signal be largest? Smallest? Hint: compute the LOS unit vector and dot it with the dominant displacement direction.


## Part IV: Non-uniform slip distributions

### What does the surface see?

Real earthquakes do not slip uniformly — slip concentrates on **asperities** and may be shallow, deep, bilateral, or unilateral. In this part you will prescribe different slip distributions and observe how they appear at the surface. The key question: **how much information about the subsurface slip distribution is preserved in the surface observations?**


In [ ]:
def plot_slip_and_surface(slip_2d, title='', fault_params=None, grid_km=70):
    """
    Plot a prescribed slip distribution on the fault and the resulting
    surface displacement and synthetic InSAR interferogram.

    Parameters
    ----------
    slip_2d      : (n_along, n_down) array  Slip in meters on each patch
    fault_params : dict  Fault geometry parameters
    """
    if fault_params is None:
        fault_params = dict(strike_deg=322, dip_deg=85, rake_deg=180,
                             length_km=50, width_km=16, depth_top_km=0)
    n_along, n_down = slip_2d.shape

    tris, slips, centers = make_fault_mesh(
        **fault_params, n_along=n_along, n_down=n_down, slip_m=slip_2d
    )
    obs, X_km, Y_km = make_obs_grid(grid_km=grid_km, nx=180)
    disp = hs.disp_free(obs, tris, slips, nu=NU)

    # LOS displacement (Sentinel-1 ascending)
    los = np.array([-np.sin(np.radians(38))*np.sin(np.radians(-12)),
                    -np.sin(np.radians(38))*np.cos(np.radians(-12)),
                     np.cos(np.radians(38))])
    los_m   = (disp @ los).reshape(X_km.shape)
    phase   = (4*np.pi/WAVELENGTH) * los_m
    wrapped = np.angle(np.exp(1j*phase))

    # Horizontal magnitude
    horiz = np.sqrt(disp[:,0]**2 + disp[:,1]**2).reshape(X_km.shape)*100
    vert  = disp[:,2].reshape(X_km.shape)*100

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Panel 1: slip distribution
    ax = axes[0]
    # Fault coordinates for display
    along_km = np.arange(n_along) * fault_params['length_km']/n_along
    down_km  = np.arange(n_down)  * fault_params['width_km']/n_down
    DA, DL   = np.meshgrid(down_km, along_km)
    c0 = ax.pcolormesh(DL, DA, slip_2d, cmap='hot_r',
                        vmin=0, vmax=slip_2d.max())
    plt.colorbar(c0, ax=ax, label='Slip (m)')
    ax.set_xlabel('Along-strike distance (km)')
    ax.set_ylabel('Down-dip distance (km)')
    ax.invert_yaxis()
    ax.set_title('Prescribed slip\n(on fault plane)')
    ax.set_aspect('equal')

    # Panel 2: vertical surface displacement
    ax = axes[1]
    lim = max(np.percentile(np.abs(vert), 98), 0.1)
    c1  = ax.contourf(X_km, Y_km, vert, levels=50, cmap='RdBu_r',
                       vmin=-lim, vmax=lim)
    plt.colorbar(c1, ax=ax, label='Vertical disp. (cm)')
    ax.set_title('Vertical surface displacement')
    ax.set_xlabel('East (km)'); ax.set_ylabel('North (km)')
    ax.set_aspect('equal')

    # Panel 3: synthetic InSAR
    ax = axes[2]
    ax.contourf(X_km, Y_km, wrapped, levels=128, cmap='hsv',
                 vmin=-np.pi, vmax=np.pi)
    ax.set_title(f'Synthetic InSAR (ascending)\neach fringe = {WAVELENGTH/2*100:.1f} cm LOS')
    ax.set_xlabel('East (km)'); ax.set_ylabel('North (km)')
    ax.set_aspect('equal')

    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

    return disp, X_km, Y_km


# ── Case 1: Uniform slip ─────────────────────────────────────────────────────
n_a, n_d = 8, 5
slip_uniform = np.ones((n_a, n_d)) * 2.0
_ = plot_slip_and_surface(slip_uniform, title='Case 1: Uniform slip (2 m everywhere)')


In [ ]:
# ── Case 2: Gaussian asperity ────────────────────────────────────────────────
n_a, n_d = 8, 5
ci, cj   = n_a/2 - 0.5, n_d/2 - 0.5  # center of fault
I, J = np.meshgrid(np.arange(n_a), np.arange(n_d), indexing='ij')
slip_gauss = 4.0 * np.exp(-((I-ci)**2/4 + (J-cj)**2/1.5))

_ = plot_slip_and_surface(slip_gauss,
    title='Case 2: Gaussian asperity (peak 4 m, centered)')


In [ ]:
# ── Case 3: Shallow vs. deep slip ────────────────────────────────────────────
n_a, n_d = 8, 5

# Shallow: slip concentrated in top two rows
slip_shallow = np.zeros((n_a, n_d))
slip_shallow[:, :2] = 3.0

# Deep: slip concentrated in bottom two rows
slip_deep = np.zeros((n_a, n_d))
slip_deep[:, 3:] = 3.0

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for row_idx, (slip_2d, label) in enumerate([
        (slip_shallow, 'Shallow slip (top 40% of fault)'),
        (slip_deep,    'Deep slip (bottom 40% of fault)')]):

    fault_params = dict(strike_deg=322, dip_deg=85, rake_deg=180,
                         length_km=50, width_km=16, depth_top_km=0)
    tris, slips, _ = make_fault_mesh(**fault_params, n_along=n_a, n_down=n_d,
                                      slip_m=slip_2d)
    obs, X_km, Y_km = make_obs_grid(grid_km=70, nx=180)
    disp = hs.disp_free(obs, tris, slips, nu=NU)
    los  = np.array([-np.sin(np.radians(38))*np.sin(np.radians(-12)),
                     -np.sin(np.radians(38))*np.cos(np.radians(-12)),
                      np.cos(np.radians(38))])
    los_m   = (disp @ los).reshape(X_km.shape)
    wrapped = np.angle(np.exp(1j*(4*np.pi/WAVELENGTH)*los_m))
    vert    = disp[:,2].reshape(X_km.shape)*100

    # Slip panel
    ax = axes[row_idx, 0]
    ax.pcolormesh(np.arange(n_a)*50/n_a, np.arange(n_d)*16/n_d,
                   slip_2d.T, cmap='hot_r', vmin=0, vmax=3)
    ax.set_title(f'Slip: {label}', fontsize=9)
    ax.set_xlabel('Along-strike (km)'); ax.set_ylabel('Down-dip (km)')
    ax.invert_yaxis()

    # Vertical displacement
    ax = axes[row_idx, 1]
    lim = max(np.percentile(np.abs(vert), 98), 0.5)
    ax.contourf(X_km, Y_km, vert, levels=40, cmap='RdBu_r', vmin=-lim, vmax=lim)
    ax.set_title('Vertical displacement (cm)', fontsize=9)
    ax.set_aspect('equal'); ax.set_xlabel('East (km)')

    # InSAR
    ax = axes[row_idx, 2]
    ax.contourf(X_km, Y_km, wrapped, levels=128, cmap='hsv',
                 vmin=-np.pi, vmax=np.pi)
    ax.set_title('Synthetic InSAR', fontsize=9)
    ax.set_aspect('equal'); ax.set_xlabel('East (km)')

plt.suptitle('Shallow vs. deep slip — same total moment, different surface signature',
             fontsize=11)
plt.tight_layout()
plt.show()


> **Non-uniform slip questions:**
> 1. The Gaussian asperity and the uniform slip case have different total seismic moments. Compute the moment $M_0 = \mu \sum_k A_k s_k$ for each, where $A_k$ is the area of patch $k$ and $s_k$ is its slip. How do they compare?
> 2. Compare the **shallow** and **deep** slip interferograms. Can you tell from the InSAR alone which is which? What features in the fringe pattern distinguish them?
> 3. The shallow slip produces a much higher peak fringe density near the fault than the deep slip. Explain why in terms of the Green's function dependence on depth.
> 4. Suppose you have only far-field GNSS stations (>50 km from the fault). Can you distinguish the shallow from the deep slip distribution? What does this tell you about the importance of near-field observations?
> 5. This is the fundamental **non-uniqueness problem** in geodetic inversion — different slip distributions can produce similar surface observations. Sketch two different slip distributions (on paper) that would produce nearly identical surface displacement at distances >30 km from the fault. What additional constraint could distinguish them?


## Part V: Forward prediction for Ridgecrest

### Building the Green's function matrix

Now we assemble everything into the matrix equation $\mathbf{d} = \mathbf{G}\mathbf{s}$ for the Ridgecrest M7.1. We will:

1. Build the fault mesh using the published geometry
2. Compute the Green's function matrix for GNSS stations
3. Load the real coseismic GNSS offsets from Lab 4
4. Compute the forward prediction for a uniform slip model
5. Compute residuals and identify where the simple model breaks down

This is the setup for the inversion lab — next week we will solve for the slip distribution that minimizes these residuals.

Coseismic offsets: [Nevada Geodetic Laboratory, UNR](https://geodesy.unr.edu/news_items/20190707/ci38457511_forweb.txt)  
Course copy: [datasets/Ridgecrest_coseismic/ci38457511_forweb.txt](https://github.com/amtseismo/EPS166/tree/main/datasets/Ridgecrest_coseismic)


In [ ]:
# ── Load Ridgecrest GNSS coseismic offsets ────────────────────────────────────
OFFSETS_URL = (
    "https://raw.githubusercontent.com/amtseismo/EPS166/main/"
    "datasets/Ridgecrest_coseismic/ci38457511_forweb.txt"
)

def load_gnss_offsets(url):
    with urllib.request.urlopen(url) as r:
        text = r.read().decode()
    records = []
    for line in text.strip().split('\n'):
        if line.strip() and not line.startswith('=') and not line.startswith('Sta'):
            p = line.split()
            if len(p) >= 9:
                records.append({'station': p[0],
                                'lon': float(p[1]), 'lat': float(p[2]),
                                'de': float(p[3])*1000, 'dn': float(p[4])*1000,
                                'du': float(p[5])*1000,
                                'sde': float(p[6])*1000, 'sdn': float(p[7])*1000,
                                'sdu': float(p[8])*1000})
    return pd.DataFrame(records)


offsets = load_gnss_offsets(OFFSETS_URL)
print(f'Loaded {len(offsets)} GNSS stations')

# Select stations within 150 km of epicenter
EQ_LAT, EQ_LON = 35.770, -117.599
offsets['dist_km'] = np.sqrt(
    ((offsets.lat - EQ_LAT)*111)**2 +
    ((offsets.lon - EQ_LON)*111*np.cos(np.radians(EQ_LAT)))**2
)
near = offsets[offsets.dist_km < 150].copy()
print(f'{len(near)} stations within 150 km')
display(near[['station','lon','lat','de','dn','du','dist_km']].head(10))


In [ ]:
# ── Convert GNSS coordinates to fault-centered Cartesian ─────────────────────
def lonlat_to_xy(lon, lat, origin_lon, origin_lat):
    """Simple local Cartesian coordinates in meters."""
    scale = 111195.0  # m per degree latitude
    x = (lon - origin_lon) * scale * np.cos(np.radians(origin_lat))
    y = (lat - origin_lat) * scale
    return x, y


near = near.copy()
near['x_m'], near['y_m'] = lonlat_to_xy(
    near.lon.values, near.lat.values, EQ_LON, EQ_LAT
)

# ── Build GNSS observation positions ─────────────────────────────────────────
gnss_obs = np.column_stack([
    near.x_m.values, near.y_m.values, np.zeros(len(near))
])

# ── Build Ridgecrest fault mesh ───────────────────────────────────────────────
# Published geometry: Liu et al. (2019)
RC_FAULT = dict(strike_deg=322, dip_deg=85, rake_deg=180,
                 length_km=50, width_km=16, depth_top_km=0)
N_ALONG, N_DOWN = 8, 5  # 40 patches

# ── Build Green's function matrix for GNSS ───────────────────────────────────
# G shape: (3*n_stations, n_patches)  — 3 components per station
n_sta    = len(near)
n_patch  = N_ALONG * N_DOWN
G_gnss   = np.zeros((3 * n_sta, n_patch))

along, downdip = fault_unit_vectors(RC_FAULT['strike_deg'], RC_FAULT['dip_deg'])
r_rak = np.radians(RC_FAULT['rake_deg'])
dl    = RC_FAULT['length_km']*1e3 / N_ALONG
dw    = RC_FAULT['width_km']*1e3  / N_DOWN
C0    = np.array([0., 0., -RC_FAULT['depth_top_km']*1e3])

print(f'Building Green\'s function matrix: {3*n_sta} obs × {n_patch} patches...')

patch_centers = []
for k, (i, j) in enumerate([(i,j) for i in range(N_ALONG) for j in range(N_DOWN)]):
    corner = C0 + i*dl*along + j*dw*downdip
    P = [corner, corner+dl*along,
         corner+dl*along+dw*downdip, corner+dw*downdip]
    tris_k  = np.array([[P[0],P[1],P[2]],[P[0],P[2],P[3]]], dtype=float)
    slips_k = np.tile([np.cos(r_rak), -np.sin(r_rak), 0.], (2,1))
    disp_k  = hs.disp_free(gnss_obs, tris_k, slips_k, nu=NU)  # (n_sta, 3)
    G_gnss[:, k] = disp_k.ravel()   # stack E,N,U for all stations
    patch_centers.append(corner + 0.5*dl*along + 0.5*dw*downdip)

patch_centers = np.array(patch_centers)
print(f'G_gnss shape: {G_gnss.shape}')

# ── Forward prediction for uniform slip (1.5 m) ──────────────────────────────
s_uniform = np.ones(n_patch) * 1.5  # m
d_pred    = G_gnss @ s_uniform       # predicted displacements (m)
print(f'Max predicted E offset: {np.abs(d_pred[0::3]).max()*1000:.0f} mm')
print(f'Max predicted N offset: {np.abs(d_pred[1::3]).max()*1000:.0f} mm')


In [ ]:
# ── Compare forward model to observed offsets ─────────────────────────────────
de_obs  = near.de.values / 1000   # mm → m
dn_obs  = near.dn.values / 1000
de_pred = d_pred[0::3]
dn_pred = d_pred[1::3]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

scale = 0.02  # degrees per meter for quiver arrows

for ax, de_p, dn_p, title in zip(
        axes,
        [de_obs, de_pred],
        [dn_obs, dn_pred],
        ['Observed GNSS offsets', 'Forward model (uniform 1.5 m slip)']):

    mag = np.sqrt(de_p**2 + dn_p**2)
    sc  = ax.scatter(near.lon, near.lat, c=mag*1000, cmap='plasma',
                      s=30, zorder=4, vmin=0)
    plt.colorbar(sc, ax=ax, label='Horizontal magnitude (mm)')
    ax.quiver(near.lon, near.lat, de_p*scale, dn_p*scale,
              scale=1, scale_units='xy', angles='xy', color='k',
              width=0.003, zorder=5)
    ax.plot(EQ_LON, EQ_LAT, 'r*', ms=15, zorder=6, label='Epicenter')
    # Approximate fault trace
    s = np.radians(322)
    dx = np.sin(s)*0.25 / np.cos(np.radians(EQ_LAT))
    dy = np.cos(s)*0.25
    ax.plot([EQ_LON-dx, EQ_LON+dx], [EQ_LAT-dy, EQ_LAT+dy],
            'k-', lw=2, label='M7.1 fault trace')
    ax.set_aspect(1/np.cos(np.radians(EQ_LAT)))
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.suptitle('Ridgecrest M7.1 — observed vs. forward model GNSS offsets\n'
             '(scale: 1° ≈ 50 km, arrows show horizontal displacement)',
             fontsize=11)
plt.tight_layout()
plt.show()

# Residuals
res_e = (de_obs - de_pred)*1000  # mm
res_n = (dn_obs - dn_pred)*1000
rms_e = np.sqrt(np.mean(res_e**2))
rms_n = np.sqrt(np.mean(res_n**2))
print(f'\nResiduals (observed - predicted):')
print(f'  East RMS:  {rms_e:.1f} mm')
print(f'  North RMS: {rms_n:.1f} mm')
print(f'  Max residual: {max(np.abs(res_e).max(), np.abs(res_n).max()):.1f} mm')


In [ ]:
# ── Synthetic InSAR for Ridgecrest ────────────────────────────────────────────
obs_grid, X_km, Y_km = make_obs_grid(grid_km=100, nx=250)

# Forward model on dense grid
tris_rc, slips_rc, _ = make_fault_mesh(
    **RC_FAULT, n_along=N_ALONG, n_down=N_DOWN,
    slip_m=np.ones((N_ALONG, N_DOWN))*1.5
)
disp_grid = hs.disp_free(obs_grid, tris_rc, slips_rc, nu=NU)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, head_deg, label in zip(axes, [-12., 192.],
                                 ['Ascending (heading≈-12°)',
                                  'Descending (heading≈192°)']):
    inc = np.radians(38); head = np.radians(head_deg)
    los = np.array([-np.sin(inc)*np.sin(head),
                    -np.sin(inc)*np.cos(head),
                     np.cos(inc)])
    los_m   = (disp_grid @ los).reshape(X_km.shape)
    wrapped = np.angle(np.exp(1j*(4*np.pi/WAVELENGTH)*los_m))

    ax.contourf(X_km, Y_km, wrapped, levels=128, cmap='hsv',
                 vmin=-np.pi, vmax=np.pi)
    # Fault trace
    s_rad = np.radians(322)
    flen  = 25  # km
    ax.plot([-np.sin(s_rad)*flen, np.sin(s_rad)*flen],
            [-np.cos(s_rad)*flen, np.cos(s_rad)*flen],
            'k-', lw=2.5, label='M7.1 fault trace')
    ax.set_title(f'Synthetic interferogram — {label}\n'
                  f'Uniform 1.5 m slip, {WAVELENGTH/2*100:.1f} cm/fringe')
    ax.set_xlabel('East (km)'); ax.set_ylabel('North (km)')
    ax.set_aspect('equal')
    ax.legend(fontsize=8)

plt.suptitle('Ridgecrest M7.1 — synthetic InSAR for uniform slip forward model',
             fontsize=11)
plt.tight_layout()
plt.show()


> **Ridgecrest forward model questions:**
> 1. The RMS residual between the observed and predicted GNSS offsets is printed above. Is this large or small relative to the observed displacements? What does the residual pattern tell you about where the uniform slip model fails?
> 2. Look at the largest residuals. Are they near the fault or far from it? What does this suggest about where the slip distribution departs most from uniform?
> 3. The Green's function matrix $\mathbf{G}$ has shape `(3*n_stations, n_patches)`. What does each column represent physically? What does each row represent?
> 4. We used 8×5=40 patches. If we used 16×10=160 patches (finer mesh), how would $\mathbf{G}$ change in shape? Would the forward prediction for uniform slip change?
> 5. Compare your synthetic ascending and descending interferograms. On which side of the fault is the signal larger for ascending? For descending? Is this consistent with what you found in the InSAR lab?
> 6. In the inversion lecture next week, we will solve $\mathbf{s} = (\mathbf{G}^T\mathbf{G})^{-1}\mathbf{G}^T\mathbf{d}$ for the slip distribution. Based on the residuals you computed here, do you expect the inverted slip to be larger or smaller than 1.5 m, and where on the fault?


## Synthesis

Write a short paragraph (3–5 sentences) answering each of the following.

> **1. The Green's function concept**  
> Explain in your own words what a Green's function is in the context of elastic dislocation modeling. Why is the linearity of the elastic equations so important? If the crust were nonlinear (e.g., if stress–strain were not proportional), how would this change the forward modeling and inversion problem?

> **2. Depth and resolution**  
> Based on what you observed in Parts II and IV, explain the relationship between fault depth and the spatial resolution of the surface displacement signal. A geodetic network with stations spaced 20 km apart would struggle to resolve slip variations at what depth scale? What could you do to improve resolution of deep slip?

> **3. The limits of the forward model**  
> The uniform slip forward model for Ridgecrest leaves significant residuals. List three physical factors that the homogeneous elastic half-space model ignores, and for each one explain whether you would expect it to matter for the Ridgecrest M7.1.

> **4. From forward model to inversion**  
> The forward problem ($\mathbf{d} = \mathbf{G}\mathbf{s}$, given $\mathbf{s}$, find $\mathbf{d}$) and the inverse problem (given $\mathbf{d}$, find $\mathbf{s}$) are mathematically related but very different in practice. Based on what you saw in this lab, explain why the inverse problem is harder than the forward problem. What additional information or constraints are needed to make the inversion well-posed?


## Summary

- A rectangular fault is represented in `cutde` as **two triangles** sharing a diagonal. The corner coordinates are computed from strike, dip, and depth using along-strike and down-dip unit vectors. The same framework handles vertical strike-slip faults, dipping thrust faults, and shallow subduction interfaces.

- A **Green's function** is the surface displacement due to unit slip on a single fault patch. It is localized, decays with distance, and broadens with depth. Deeper patches produce broader, smoother surface signals — this is why deep slip is harder to resolve from surface observations.

- The `cutde`/Okada forward model recovers the **Savage–Burford** interseismic profile and the **1D coseismic** profile exactly in the appropriate limits — confirming that these familiar results are limiting cases of the same elastic dislocation framework.

- **Non-uniform slip** produces surface displacement that is a weighted superposition of Green's functions. Different slip distributions can produce similar far-field signals — this is the **non-uniqueness problem** that regularization in the inversion must address.

- The **Green's function matrix** $\mathbf{G}$ (shape: observations × patches) encodes the full forward model. The data vector $\mathbf{d} = \mathbf{G}\mathbf{s}$ is a simple matrix–vector product. Different data types (GNSS, InSAR ascending, InSAR descending) contribute different rows to $\mathbf{G}$ with corresponding rows in $\mathbf{d}$.

- The **uniform slip forward model** for Ridgecrest explains the large-scale pattern of GNSS offsets but leaves significant residuals, especially near the fault. These residuals encode information about the true (non-uniform) slip distribution that the inversion will recover.

### References

- Okada, Y. (1985). Surface deformation due to shear and tensile faults in a half-space. *BSSA*, 75(4), 1135–1154. https://doi.org/10.1785/BSSA0750041135
- Meade, B. J. (2007). Algorithms for triangular dislocation elements. *Computers & Geosciences*, 33(8), 1064–1075. https://doi.org/10.1016/j.cageo.2006.12.003
- Liu, C., et al. (2019). Fault geometry and slip distribution of the 2019 Ridgecrest earthquake sequence. *GRL*, 46. https://doi.org/10.1029/2019GL084949
- Nevada Geodetic Laboratory, UNR. http://geodesy.unr.edu
- cutde library: https://github.com/cutde-org/cutde
